# Module 06: Error Handling & Logging

Build robust ML pipelines that gracefully handle failures and provide clear debugging information.

**ML Focus:**
- Robust data loading with fallbacks
- Logging training progress
- Debugging data quality issues
- Custom exceptions for ML domain errors

## 1. try/except/else/finally — The Full Pattern

The complete error handling structure with all four blocks.

In [ ]:
# Demonstrate try/except/else/finally
def divide_numbers(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Error: Cannot divide by zero')
        return None
    else:
        print('Division succeeded')
        return result
    finally:
        print('Cleanup: this always runs')

print('Test 1: divide 10 by 2')
print('Result:', divide_numbers(10, 2))
print()
print('Test 2: divide 10 by 0')
print('Result:', divide_numbers(10, 0))

## 2. Catching Specific Exceptions

Always catch specific exception types. A bare `except:` catches everything including KeyboardInterrupt.

In [ ]:
# Simulate ML data loading with specific error handling
import json

def load_ml_config(path):
    try:
        with open(path, 'r') as f:
            config = json.load(f)
        print('Loaded config with keys:', list(config.keys()))
        return config
    except FileNotFoundError:
        print('Error: Config file not found —', path)
        print('Using default configuration instead')
        return {'learning_rate': 0.01, 'epochs': 10}
    except json.JSONDecodeError:
        print('Error: Invalid JSON in config file')
        return None

# Test with missing file
config = load_ml_config('nonexistent_config.json')
print()

# Test with valid file
with open('good_config.json', 'w') as f:
    json.dump({'learning_rate': 0.001, 'epochs': 50}, f)
config = load_ml_config('good_config.json')

## 3. Custom Exceptions for ML Pipelines

Define domain-specific exceptions that carry semantic meaning.

In [ ]:
class DataValidationError(Exception):
    """Raised when data fails validation checks."""
    pass

class ModelConvergenceError(Exception):
    """Raised when model fails to converge."""
    pass

def validate_dataset(features, labels):
    if len(features) == 0:
        raise DataValidationError('Empty feature set')
    if len(features) != len(labels):
        raise DataValidationError(
            'Feature count ' + str(len(features))
            + ' does not match label count ' + str(len(labels))
        )
    print('Dataset validated:', len(features), 'samples')

# Test custom exceptions
try:
    validate_dataset([1, 2, 3], [0, 1])
except DataValidationError as e:
    print('Caught custom exception:', e)

## 4. The logging Module — Basic Setup

Replace `print()` with `logging` for structured, timestamped, level-based output.

In [ ]:
import logging

# Basic configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger('ml_pipeline')

logger.debug('This is debug — not shown at INFO level')
logger.info('Starting ML pipeline')
logger.warning('Low memory — consider reducing batch size')
logger.error('Failed to load validation set')

## 5. Advanced Logging: Handlers and Formatters

Send different levels to console vs file. Useful for monitoring training runs.

In [ ]:
import logging

# Create logger
logger = logging.getLogger('training')
logger.setLevel(logging.DEBUG)

# Console handler — only INFO and above
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console_format = logging.Formatter('%(levelname)s: %(message)s')
console.setFormatter(console_format)

# File handler — everything including DEBUG
file_handler = logging.FileHandler('training.log', mode='w')
file_handler.setLevel(logging.DEBUG)
file_format = logging.Formatter(
    '%(asctime)s | %(levelname)-8s | %(message)s'
)
file_handler.setFormatter(file_format)

# Add handlers
logger.addHandler(console)
logger.addHandler(file_handler)

# Test
logger.debug('Initializing model parameters')
logger.info('Epoch 1/10 — loss: 0.5234')
logger.info('Epoch 2/10 — loss: 0.3891')
logger.warning('Loss plateau detected — consider reducing LR')
logger.debug('Gradient norm: 0.042')

print('\nCheck training.log for full debug output')

## 6. assert Statements for Sanity Checks

Use `assert` for debugging and development. For production, use explicit `if` + `raise`.

In [ ]:
# Training function with assertions
def train_epoch(model, features, labels):
    # Sanity checks
    assert len(features) > 0, 'Empty feature set'
    assert len(features) == len(labels), 'Mismatched sizes'
    
    # Simulate training
    loss = 0.5  # pretend we trained
    
    # Output check
    assert loss >= 0, 'Loss should be non-negative'
    assert loss == loss, 'Loss is NaN'  # NaN != NaN
    
    return loss

# Valid case
loss = train_epoch('model', [1, 2, 3], [0, 1, 0])
print('Training loss:', loss)

print()
# AssertionError case
try:
    train_epoch('model', [1, 2], [0, 1, 0])
except AssertionError as e:
    print('Caught assertion:', e)

## 7. Robust ML Pipeline Pattern

Combine custom exceptions, logging, and error handling into a complete ML pipeline.

In [ ]:
import logging
import json
import time

# Setup logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger('pipeline')

class PipelineError(Exception):
    pass

def run_pipeline(config_path):
    logger.info('Pipeline started')
    start_time = time.time()
    
    try:
        # Step 1: Load config
        logger.info('Step 1: Loading config from %s', config_path)
        with open(config_path, 'r') as f:
            config = json.load(f)
        logger.info('Config loaded: %s', config)
        
        # Step 2: Load data
        logger.info('Step 2: Loading data')
        data = [1, 2, 3, 4, 5]  # simulated
        logger.info('Data loaded: %d samples', len(data))
        
        # Step 3: Train
        logger.info('Step 3: Training for %d epochs', config.get('epochs', 10))
        for epoch in range(config.get('epochs', 10)):
            logger.debug('Epoch %d complete', epoch + 1)
        
        elapsed = time.time() - start_time
        logger.info('Pipeline completed in %.2f seconds', elapsed)
        
    except FileNotFoundError:
        logger.error('Config file not found: %s', config_path)
        raise PipelineError('Configuration missing')
    except json.JSONDecodeError as e:
        logger.error('Invalid JSON config: %s', e)
        raise PipelineError('Bad configuration')
    except Exception as e:
        logger.critical('Unexpected pipeline failure: %s', e)
        raise

# Run with good config
with open('pipeline_config.json', 'w') as f:
    json.dump({'epochs': 3, 'batch_size': 32}, f)

run_pipeline('pipeline_config.json')

## Summary

**Key takeaways:**
1. Use `try/except/else/finally` for complete error handling
2. Catch specific exceptions, never bare `except:`
3. Create custom exceptions for domain-specific failures
4. Use `logging` with timestamps, levels, and file output
5. Use `assert` for development sanity checks
6. Combine patterns into robust ML pipelines